### 문장을 벡터로 변환
- 문장을 숫자 배열로 변경
- 벡터 : 숫자 배열
- 차원 : 벡터 개수

| 문장 | 느리고 | 배송 | 별로 | 빠르고 | 좋아요 | 최고 |
|--|--|--|--|--|--|--|
|배송 빠르고 좋아요 | 0 | 1 | 0 | 1 | 1 | 0 |
|배송 느리고 별로 | 1 | 1 | 1 | 0 | 0 | 0 |
|좋아요 좋아요 최고| 0 | 0 | 0 | 0 | 2 | 1 |

In [1]:
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
docs = ['배송 빠르고 좋아요','배송 느리고 별로','좋아요 좋아요 최고']

vec = CountVectorizer()
X = vec.fit_transform(docs)       # fit: 단어장 만들기 / transform : 문장을 숫자로 변경
X

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 8 stored elements and shape (3, 6)>

In [3]:
# 단어장 확인
vec.get_feature_names_out()

array(['느리고', '배송', '별로', '빠르고', '좋아요', '최고'], dtype=object)

In [4]:
X.toarray()

array([[0, 1, 0, 1, 1, 0],
       [1, 1, 1, 0, 0, 0],
       [0, 0, 0, 0, 2, 1]])

In [5]:
X.shape

(3, 6)

#### 네이버 영화 리뷰 데이터

In [ ]:
# from google.colab import drive
# drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [11]:
# github 네이버 영화리뷰 데이터 가져오기

import pandas as pd

url = 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt'
nsmc = pd.read_csv(url, sep='\t')
nsmc.shape

(150000, 3)

In [12]:
nsmc.head(3)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0


In [18]:
nsmc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [21]:
# null 데이터 제거

nsmc = nsmc.dropna()

# 중복 리뷰 데이터 제거
nsmc = nsmc.drop_duplicates('document')

In [22]:
nsmc.info()

<class 'pandas.core.frame.DataFrame'>
Index: 146182 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        146182 non-null  int64 
 1   document  146182 non-null  object
 2   label     146182 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.5+ MB


In [25]:
# document => text    컬럼명 변경

# nsmc.columns = ['id', 'document', 'label']
nsmc = nsmc.rename(columns={"document":"text"})[['text','label']]
nsmc.head(3)

,text,label
0,아 더빙.. 진짜 짜증나네요 목소리,0
1,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,너무재밓었다그래서보는것을추천한다,0


In [27]:
# 임의로 긍정, 부정 각 1000개 추출 => 2000개
df = nsmc.groupby('label').sample(1000, random_state=42)
df['label'].value_counts()

,count
label,
0,1000
1,1000


In [ ]:
# 저장하기

df.to_csv("../data/reviews.csv", index=False, encoding="utf-8-sig")

#### data/reviews.csv 가져오기

In [ ]:
review_df = pd.read_csv("../data/reviews.csv", encoding="utf-8-sig")
review_df.shape

(2000, 2)

In [36]:
review_df.head(2)

,text,label
0,설정이 복잡하고 캐릭터가 죽어 있어 마지막 전에 질린다,0
1,줄거리 진짜 꽝... 슈퍼가 보는맛에 버텼다...,0


In [38]:
# 벡터 추출

vec = CountVectorizer()
X = vec.fit_transform(review_df['text'])       # fit: 단어장 만들기 / transform : 문장을 숫자로 변경
y = review_df['label']

In [39]:
X.shape

(2000, 9968)

In [41]:
# 단어장 보기

vec.get_feature_names_out(X)

array(['007', '01임', '04년도', ..., '힘이', '힘이나는', '힙합한'], dtype=object)

In [43]:
# 벡터
X.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [45]:
# 단어별 전체 등장 횟수

# 문서 전체에서 각 단어가 등장한 횟수 계산 후 결과를 1차원 배열로 변환
counts = X.sum(axis=0).A1
print((counts == 1).sum(), "개 단어가 한번만 등장")
print(y.value_counts())

8346 개 단어가 한번만 등장
label
0    1000
1    1000
Name: count, dtype: int64


#### 학습시키기

In [46]:
from sklearn.linear_model import LogisticRegression

# 분류모델
model = LogisticRegression()

# 학습(문장 벡터, 정답)
model.fit(X,y)

LogisticRegression()

#### 모델의 학습내용 확인

In [51]:
model.coef_[0]

array([ 0.39189174, -0.07446737, -0.14791447, ...,  0.26213461,
        0.01589909, -0.06546688])

In [65]:
weights = pd.Series(model.coef_[0], index = vec.get_feature_names_out())
weights = weights.sort_values()

print(weights.head(5))
print(weights.tail(5))
print(model.intercept_)     # 편향(bias)

쓰레기   -1.875708
ㅡㅡ    -1.444053
이게    -1.324084
스토리   -1.168751
별로    -1.143842
dtype: float64
재밌게    1.105174
봐도     1.165308
최고의    1.389857
좋다     1.614637
최고     1.649706
dtype: float64
[-0.00087348]


In [68]:
s = "연기가 좋고 정말 재밌어요"
x = vec.transform([s]).toarray()[0]     # 기존 단어장에 새 문장을 추가

print(x)     # 이 단어장은 2000개 립로 만들었고 단어가 9968개이기 때문에 x는 항상 길이

# 벡터 내적을 구하기 : inner product를 한 다음에 편향값을 더해준다
# 편향값이란? 데이터의 기본 경향을 반영하는 것
my_score = (x * model.coef_[0]).sum() + model.intercept_[0]
my_score


[0 0 0 ... 0 0 0]


np.float64(2.0141738861686944)

In [69]:
# 모델이 계산한 점수와 비교

model_score = model.decision_function(vec.transform([s]))[0]
model_score

np.float64(2.014173886168694)

#### 학습완료 == 가중치 벡터와 편향의 값이 정해졌다

#### 테스트 - 새 문장 추론

In [70]:
words = ['연기가 좋고 정말 재밌어요', '지루하고 별로였다','배송이 빠르고 포장이 꼼꼼해요']

# 새 문장도 같은 단어장으로 벡터화
X_new = vec.transform(words)

# 예측 0 or 1
print(model.predict(X_new))
# [부정확률, 긍정확률]
print(model.predict_proba(X_new).round(2))

[1 0 0]
[[0.12 0.88]
 [0.68 0.32]
 [0.5  0.5 ]]


In [73]:
s = "배송이 빠르고 포장이 꼼꼼해요"
x = vec.transform([s]).toarray().sum()
print(x)
model_score = model.decision_function(vec.transform([s]))[0]
print(model_score)
print(model.predict_proba(vec.transform([s])))

0
-0.0008734769421811165
[[0.50021837 0.49978163]]


In [ ]:
import joblib, os

# 단어장, 몯ㄹ 저장
joblib.dump((vec, model), "../data/sentiment.joblib")

print(os.path.getsize("../data/sentiment.joblib"), "bytes")

258736 bytes


In [ ]:
# 저장된 joblib 불러오기
(vec, model) = joblib.load("../data/sentiment.joblib")
print(model.predict(vec.transform(['정말 재밌어요'])))

[1]


In [78]:
# 모델의 파라미터 확인

# 가중치 개수 + 편향 개수
print(model.coef_.size + model.intercept_.size)

9969


In [2]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("../data/reviews.csv", encoding="utf-8-sig")

# 전체 데이터에서 학습 데이터와 테스트 데이터 분리
# stratify=df['label'] : 긍정, 부정 비율을 양쪽에 똑같이 유지
text_train, text_test, y_train, y_test = train_test_split(
    df['text'],df['label'], test_size=0.2, random_state=42, stratify=df['label'])

print(len(text_train), len(text_test))

1600 400


In [87]:
vec = CountVectorizer()

# 단어장 생성(학습 데이터로만) / 숫자 벡터 변환
X_train = vec.fit_transform(text_train)

# 테스트 데이터(fit => 새 단어장 만듬 => 벡터 변환)
X_test = vec.transform(text_test)


# 분류모델
model = LogisticRegression()

# 학습(문장 벡터, 정답)
model.fit(X_train,y_train)
print(f"학습 정확도 :{model.score(X_train, y_train)}")
print(f"테스트 정확도 :{model.score(X_test, y_test)}")

학습 정확도 :0.99625
테스트 정확도 :0.695
